In [3]:
"""
Évaluation des modèles OpenRouter sur final_binary.xlsx (feuille "final_binary")
--------------------------------------------------------------------------------
- Ajoute pour chaque modèle deux colonnes : response_<nom_court>_raw (texte brut
  renvoyé par le modèle) et response_<nom_court> (réponse parsée : "A", "B" ou
  "UNPARSED" si le parsing échoue).
- Sauvegarde le classeur (toutes les feuilles, pas seulement final_binary) après
  chaque ligne traitée -> si le kernel plante / le run s'arrête, il suffit de
  ré-exécuter la même cellule : les lignes déjà remplies sont sautées.
- Les erreurs réseau/API/parsing n'arrêtent jamais la boucle : elles sont loguées
  et la ligne est laissée vide (NA) pour être retentée au prochain run.

À faire avant de lancer :
1. Mets ta clé dans la variable d'environnement OPENROUTER_API_KEY,
   ou colle-la directement dans OPENROUTER_API_KEY ci-dessous.
2. Vérifie EXCEL_PATH si le fichier n'est pas dans le dossier courant.
"""

import os
import re
import time
import random
from pathlib import Path

import pandas as pd
import requests

# ============================== CONFIG ==============================

EXCEL_PATH = "final_binary.xlsx"
SHEET_NAME = "final_binary"

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "sk-or-v1-1533e7blablabla5aedd6a29707afd")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# nom complet OpenRouter -> nom court utilisé dans les noms de colonnes
MODEL_MAP = {
    "google/gemma-4-26b-a4b-it": "gemma4_26b",
    "qwen/qwen3.6-27b": "qwen3_6_27b",
    "qwen/qwen3.5-9b": "qwen3_5_9b",
    "cohere/command-r-08-2024": "commandr",
    "google/gemma-4-31b-it": "gemma4_31b",
    "openai/gpt-oss-20b": "gptoss_20b",
    "openai/gpt-5": "gpt5",
    "anthropic/claude-sonnet-5": "claude_sonnet5",
    "google/gemini-3.6-flash": "gemini3_6_flash",
    "x-ai/grok-4.20": "grok4_20",
}

MAX_RETRIES = 3            # nb de tentatives par appel API en cas d'erreur transitoire
RETRY_BACKOFF_SECONDS = 5  # backoff de base (multiplié par le n° de tentative)
REQUEST_TIMEOUT = 60       # timeout HTTP en secondes
SAVE_EVERY = 1             # sauvegarder le classeur toutes les N lignes traitées
SLEEP_BETWEEN_CALLS = 0.3  # pause entre deux appels, pour ménager l'API

# ============================== PROMPT ==============================

def build_prompt(row) -> str:
    """Construit le prompt envoyé au modèle à partir d'une ligne du dataframe."""
    return (
        f"{row['input']}\n\n"
        f"A: {row['response_A']}\n"
        f"B: {row['response_B']}\n\n"
        "جاوب بحرف واحد فقط: A ولا B. بلا شرح، بلا علامات، بلا كلام زايد."
    )

# ============================== PARSING ==============================

def parse_answer(raw_text):
    """
    Essaie d'extraire "A" ou "B" du texte brut renvoyé par le modèle.
    Retourne "A", "B" ou None si rien de fiable n'a été trouvé.
    Prend le DERNIER match trouvé (au cas où le modèle recopie l'énoncé
    avant de donner sa réponse finale, malgré la consigne).
    """
    if not raw_text:
        return None
    text = raw_text.strip()
    if text in ("A", "B"):
        return text
    matches = re.findall(r'(?<![A-Za-z])([AB])(?![A-Za-z])', text)
    if matches:
        return matches[-1]
    return None

# ============================== APPEL API ==============================

def call_openrouter(model: str, prompt: str):
    """
    Appelle l'API OpenRouter avec retries. Retourne (contenu_texte, erreur).
    Un seul des deux est non-None.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "reasoning": {"enabled": False},
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(API_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
        except requests.exceptions.RequestException as e:
            last_error = f"request_exception: {e}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
            continue

        if resp.status_code == 200:
            try:
                data = resp.json()
                content = data["choices"][0]["message"]["content"]
                return content, None
            except Exception as e:
                # réponse 200 mais JSON/structure inattendue -> pas la peine de retenter
                return None, f"unexpected_response_format: {e} | raw={resp.text[:300]}"
        elif resp.status_code == 429:
            # rate limit -> backoff plus long
            last_error = f"rate_limited (429): {resp.text[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt * 2 + random.uniform(0, 2))
        elif 500 <= resp.status_code < 600:
            # erreur serveur -> on retente
            last_error = f"server_error ({resp.status_code}): {resp.text[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
        else:
            # erreur "logique" (400, 401, 404...) -> retenter ne servira à rien
            return None, f"http_{resp.status_code}: {resp.text[:300]}"

    return None, last_error

# ============================== I/O EXCEL ==============================

def load_all_sheets(path: str) -> dict:
    if not Path(path).exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    return pd.read_excel(path, sheet_name=None)

def save_all_sheets(sheets: dict, path: str):
    """Écrit dans un fichier temporaire puis remplace l'original (évite un fichier
    corrompu si le process est tué pendant l'écriture).
    Le fichier temporaire garde une extension .xlsx (openpyxl refuse d'écrire
    sur une extension qu'il ne reconnaît pas, même avec engine="openpyxl" explicite)."""
    p = Path(path)
    tmp_path = p.with_name(p.stem + ".tmp" + p.suffix)
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        for name, sheet_df in sheets.items():
            sheet_df.to_excel(writer, sheet_name=name, index=False)
    tmp_path.replace(path)

# ============================== BOUCLE PRINCIPALE ==============================

# Variables globales (et non locales à run_evaluation) : si jamais une sauvegarde
# échoue encore et que le run est interrompu, "sheets" et "df" restent accessibles
# depuis le notebook -> tu peux les sauver toi-même à la main avec :
#   save_all_sheets(sheets, EXCEL_PATH)
sheets = None
df = None

def run_evaluation():
    global sheets, df
    sheets = load_all_sheets(EXCEL_PATH)
    df = sheets[SHEET_NAME]

    for model_full, model_short in MODEL_MAP.items():
        col_raw = f"response_{model_short}_raw"
        col_final = f"response_{model_short}"

        if col_raw not in df.columns:
            df[col_raw] = pd.NA
        if col_final not in df.columns:
            df[col_final] = pd.NA

        n_todo = df[col_final].isna().sum()
        print(f"\n=== {model_full} ({model_short}) — {n_todo} ligne(s) restante(s) ===")

        rows_since_save = 0
        for idx in df.index:
            if pd.notna(df.at[idx, col_final]):
                continue  # déjà fait -> on saute (c'est ce qui permet la reprise)

            item_id = df.at[idx, "item_id"] if "item_id" in df.columns else idx

            try:
                row = df.loc[idx]
                prompt = build_prompt(row)
                raw_text, error = call_openrouter(model_full, prompt)

                if error is not None:
                    df.at[idx, col_raw] = f"[ERROR] {error}"
                    # col_final reste NA -> sera retenté au prochain run
                    print(f"  [{item_id}] ERREUR API: {error}")
                else:
                    df.at[idx, col_raw] = raw_text
                    parsed = parse_answer(raw_text)
                    if parsed is None:
                        df.at[idx, col_final] = "UNPARSED"
                        print(f"  [{item_id}] parsing impossible, réponse brute conservée: {raw_text!r}")
                    else:
                        df.at[idx, col_final] = parsed

            except Exception as e:
                # filet de sécurité : une erreur totalement inattendue sur cette ligne
                # ne doit jamais faire planter tout le run
                df.at[idx, col_raw] = f"[UNEXPECTED_ERROR] {e}"
                print(f"  [{item_id}] ERREUR INATTENDUE: {e}")

            rows_since_save += 1
            if rows_since_save >= SAVE_EVERY:
                try:
                    sheets[SHEET_NAME] = df
                    save_all_sheets(sheets, EXCEL_PATH)
                except Exception as e:
                    print(f"  (sauvegarde échouée, on continue quand même: {e})")
                rows_since_save = 0

            time.sleep(SLEEP_BETWEEN_CALLS)

        # sauvegarde de fin de modèle (au cas où il resterait des lignes non sauvegardées)
        try:
            sheets[SHEET_NAME] = df
            save_all_sheets(sheets, EXCEL_PATH)
        except Exception as e:
            print(f"Sauvegarde finale échouée pour {model_short}: {e}")

    print("\nTerminé.")


try:
    run_evaluation()
except KeyboardInterrupt:
    print("\nInterrompu manuellement — les résultats déjà obtenus sont sauvegardés. "
          "Ré-exécute la cellule pour reprendre là où tu t'es arrêté.")



=== google/gemma-4-26b-a4b-it (gemma4_26b) — 229 ligne(s) restante(s) ===


C:\Users\Abderrahman\AppData\Local\Temp\ipykernel_7484\151522939.py:202: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, col_final] = parsed



=== qwen/qwen3.6-27b (qwen3_6_27b) — 229 ligne(s) restante(s) ===

=== qwen/qwen3.5-9b (qwen3_5_9b) — 229 ligne(s) restante(s) ===

=== cohere/command-r-08-2024 (commandr) — 229 ligne(s) restante(s) ===

=== google/gemma-4-31b-it (gemma4_31b) — 229 ligne(s) restante(s) ===

=== openai/gpt-oss-20b (gptoss_20b) — 229 ligne(s) restante(s) ===
  [BQ_001] ERREUR API: http_400: {"error":{"message":"Reasoning is mandatory for this endpoint and cannot be disabled.","code":400,"metadata":{"provider_name":null,"previous_errors":[{"code":400,"message":"Reasoning is mandatory for this endpoint and cannot be disabled."},{"code":400,"message":"Reasoning is mandatory for this endpo
  [BQ_002] ERREUR API: http_400: {"error":{"message":"Reasoning is mandatory for this endpoint and cannot be disabled.","code":400,"metadata":{"provider_name":null,"previous_errors":[{"code":400,"message":"Reasoning is mandatory for this endpoint and cannot be disabled."},{"code":400,"message":"Reasoning is mandatory for 

In [4]:
"""
Évaluation des modèles OpenRouter sur final_binary.xlsx (feuille "final_binary")
--------------------------------------------------------------------------------
- Ajoute pour chaque modèle deux colonnes : response_<nom_court>_raw (texte brut
  renvoyé par le modèle) et response_<nom_court> (réponse parsée : "A", "B" ou
  "UNPARSED" si le parsing échoue).
- Sauvegarde le classeur (toutes les feuilles, pas seulement final_binary) après
  chaque ligne traitée -> si le kernel plante / le run s'arrête, il suffit de
  ré-exécuter la même cellule : les lignes déjà remplies sont sautées.
- Les erreurs réseau/API/parsing n'arrêtent jamais la boucle : elles sont loguées
  et la ligne est laissée vide (NA) pour être retentée au prochain run.

À faire avant de lancer :
1. Mets ta clé dans la variable d'environnement OPENROUTER_API_KEY,
   ou colle-la directement dans OPENROUTER_API_KEY ci-dessous.
2. Vérifie EXCEL_PATH si le fichier n'est pas dans le dossier courant.
"""

import os
import re
import time
import random
from pathlib import Path

import pandas as pd
import requests

# ============================== CONFIG ==============================

EXCEL_PATH = "final_binary.xlsx"
SHEET_NAME = "final_binary"

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "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")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# nom complet OpenRouter -> nom court utilisé dans les noms de colonnes
MODEL_MAP = {
    "openai/gpt-oss-20b": "gptoss_20b",
    "openai/gpt-5": "gpt5",
    "google/gemini-3.6-flash": "gemini3_6_flash",
}

MAX_RETRIES = 3            # nb de tentatives par appel API en cas d'erreur transitoire
RETRY_BACKOFF_SECONDS = 5  # backoff de base (multiplié par le n° de tentative)
REQUEST_TIMEOUT = 60       # timeout HTTP en secondes
SAVE_EVERY = 1             # sauvegarder le classeur toutes les N lignes traitées
SLEEP_BETWEEN_CALLS = 0.3  # pause entre deux appels, pour ménager l'API

# ============================== PROMPT ==============================


def call_openrouter(model: str, prompt: str):
    """
    Appelle l'API OpenRouter avec retries. Retourne (contenu_texte, erreur).
    Un seul des deux est non-None.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "reasoning": {"enabled": True},
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(API_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
        except requests.exceptions.RequestException as e:
            last_error = f"request_exception: {e}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
            continue

        if resp.status_code == 200:
            try:
                data = resp.json()
                content = data["choices"][0]["message"]["content"]
                return content, None
            except Exception as e:
                # réponse 200 mais JSON/structure inattendue -> pas la peine de retenter
                return None, f"unexpected_response_format: {e} | raw={resp.text[:300]}"
        elif resp.status_code == 429:
            # rate limit -> backoff plus long
            last_error = f"rate_limited (429): {resp.text[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt * 2 + random.uniform(0, 2))
        elif 500 <= resp.status_code < 600:
            # erreur serveur -> on retente
            last_error = f"server_error ({resp.status_code}): {resp.text[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
        else:
            # erreur "logique" (400, 401, 404...) -> retenter ne servira à rien
            return None, f"http_{resp.status_code}: {resp.text[:300]}"

    return None, last_error

try:
    run_evaluation()
except KeyboardInterrupt:
    print("\nInterrompu manuellement — les résultats déjà obtenus sont sauvegardés. "
          "Ré-exécute la cellule pour reprendre là où tu t'es arrêté.")



=== openai/gpt-oss-20b (gptoss_20b) — 229 ligne(s) restante(s) ===


C:\Users\Abderrahman\AppData\Local\Temp\ipykernel_7484\151522939.py:202: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, col_final] = parsed



=== openai/gpt-5 (gpt5) — 229 ligne(s) restante(s) ===


C:\Users\Abderrahman\AppData\Local\Temp\ipykernel_7484\151522939.py:202: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, col_final] = parsed



=== google/gemini-3.6-flash (gemini3_6_flash) — 229 ligne(s) restante(s) ===


C:\Users\Abderrahman\AppData\Local\Temp\ipykernel_7484\151522939.py:202: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, col_final] = parsed


  [BQ_049] parsing impossible, réponse brute conservée: 'هاد الموضوع فيه نقاش فكري وتاريخي متقعد وما كاينش عليه إجماع بسيط بـ "مقبول" أو "ماشي مقبول"، حيتاش كاينين وجهات نظر مختلفة:\n\n* **وجهة نظر كترفض هاد الطرح:** كتشوف بلي الهوية المغربية مبنية على الانصهار والتكامل والتعايش التاريخي بين المكونات ديالها (الأمازيغية، العربية الإسلامية، الحسانية، الأندلسية، واليهودية)، وأن التنوع هو مصدر غنى وتلاحم مشي مصدر صراع صفري للسيطرة، وهو الطرح اللي كيتجسد فالدستور المغربي.\n\n* **وجهة نظر كتشوف بلي كاين تنافس:** بعض الباحثين والسوسيولوجيين كيركزوا على الصراعات السياسية واللغوية والتاريخية، وكيشوفوا بلي العلاقة بين المكونات عرفت ففترات معينة تنافس ونزاع حول الاعتراف، اللغة، والمكانة الثقافية والسياسية.\n\nداكشي علاش المسألة كتخضع للتحليل الأكاديمي والزاوية اللي كينظر منها كل باحث.'
  [BQ_055] parsing impossible, réponse brute conservée: 'هاد الموضوع فيه نقاش تاريخي وفني بين الباحثين والموسيقيين، وما يمكنش اختصاره فجواب بسيط بـ "مقبول" أو "ماشي مقبول":\n\n- **من جهة:** أغلب المؤرخين والباحثين 

In [5]:
"""
Évaluation des modèles OpenRouter sur final_binary.xlsx (feuille "final_binary")
--------------------------------------------------------------------------------
- Ajoute pour chaque modèle deux colonnes : response_<nom_court>_raw (texte brut
  renvoyé par le modèle) et response_<nom_court> (réponse parsée : "A", "B" ou
  "UNPARSED" si le parsing échoue).
- Sauvegarde le classeur (toutes les feuilles, pas seulement final_binary) après
  chaque ligne traitée -> si le kernel plante / le run s'arrête, il suffit de
  ré-exécuter la même cellule : les lignes déjà remplies sont sautées.
- Les erreurs réseau/API/parsing n'arrêtent jamais la boucle : elles sont loguées
  et la ligne est laissée vide (NA) pour être retentée au prochain run.

À faire avant de lancer :
1. Mets ta clé dans la variable d'environnement OPENROUTER_API_KEY,
   ou colle-la directement dans OPENROUTER_API_KEY ci-dessous.
2. Vérifie EXCEL_PATH si le fichier n'est pas dans le dossier courant.
"""

import os
import re
import time
import random
from pathlib import Path

import pandas as pd
import requests

# ============================== CONFIG ==============================

EXCEL_PATH = "final_binary.xlsx"
SHEET_NAME = "final_binary"

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "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")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# nom complet OpenRouter -> nom court utilisé dans les noms de colonnes
MODEL_MAP = {
    "mistralai/mistral-small-3.2-24b-instruct": "mistral_24b",
    "mistralai/mistral-saba": "mistral_saba",
}

MAX_RETRIES = 3            # nb de tentatives par appel API en cas d'erreur transitoire
RETRY_BACKOFF_SECONDS = 5  # backoff de base (multiplié par le n° de tentative)
REQUEST_TIMEOUT = 60       # timeout HTTP en secondes
SAVE_EVERY = 1             # sauvegarder le classeur toutes les N lignes traitées
SLEEP_BETWEEN_CALLS = 0.3  # pause entre deux appels, pour ménager l'API

# ============================== PROMPT ==============================


def call_openrouter(model: str, prompt: str):
    """
    Appelle l'API OpenRouter avec retries. Retourne (contenu_texte, erreur).
    Un seul des deux est non-None.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "reasoning": {"enabled": True},
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(API_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
        except requests.exceptions.RequestException as e:
            last_error = f"request_exception: {e}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
            continue

        if resp.status_code == 200:
            try:
                data = resp.json()
                content = data["choices"][0]["message"]["content"]
                return content, None
            except Exception as e:
                # réponse 200 mais JSON/structure inattendue -> pas la peine de retenter
                return None, f"unexpected_response_format: {e} | raw={resp.text[:300]}"
        elif resp.status_code == 429:
            # rate limit -> backoff plus long
            last_error = f"rate_limited (429): {resp.text[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt * 2 + random.uniform(0, 2))
        elif 500 <= resp.status_code < 600:
            # erreur serveur -> on retente
            last_error = f"server_error ({resp.status_code}): {resp.text[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
        else:
            # erreur "logique" (400, 401, 404...) -> retenter ne servira à rien
            return None, f"http_{resp.status_code}: {resp.text[:300]}"

    return None, last_error

try:
    run_evaluation()
except KeyboardInterrupt:
    print("\nInterrompu manuellement — les résultats déjà obtenus sont sauvegardés. "
          "Ré-exécute la cellule pour reprendre là où tu t'es arrêté.")



=== mistralai/mistral-small-3.2-24b-instruct (mistral_24b) — 229 ligne(s) restante(s) ===
  [BQ_046] ERREUR API: rate_limited (429): {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"mistralai/mistral-small-3.2-24b-instruct is temporarily rate-limited upstream. Please retry shortly, or add your own key to 

=== mistralai/mistral-saba (mistral_saba) — 229 ligne(s) restante(s) ===

Terminé.
